# LLM Training Pipeline on Google Colab (Optimized)

This notebook is tuned for **Colab Free (Tesla T4)** with fast setup, safer resume behavior, and better throughput/memory defaults.

Pipeline:
- Environment + repository setup
- Data preparation (`.txt`, `.md`, `.pdf`, `.epub`, subtitles)
- Pretraining
- SFT
- Chat inference

In [ ]:
# @title 1) Mount Drive + Runtime Sanity Check
from google.colab import drive
import os
import subprocess
import torch

drive.mount('/content/drive')

assert torch.cuda.is_available(), 'GPU runtime not detected. In Colab: Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA available:', torch.cuda.is_available())
print('Torch:', torch.__version__)

BASE_DRIVE = '/content/drive/MyDrive/llm_scratch'
PRETRAIN_DIR = f'{BASE_DRIVE}/pretrain_checkpoints'
SFT_DIR = f'{BASE_DRIVE}/sft_checkpoints'
DATA_DIR = f'{BASE_DRIVE}/training_data'

for d in [BASE_DRIVE, PRETRAIN_DIR, SFT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive dirs ready:')
print(' -', PRETRAIN_DIR)
print(' -', SFT_DIR)
print(' -', DATA_DIR)

In [ ]:
# @title 2) Clone/Update Repository
REPO_URL = 'https://github.com/Xtuber14/LLM-Training.git'  # @param {type:'string'}
REPO_DIR = '/content/LLM-Training'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git pull --ff-only

%cd {REPO_DIR}
!git rev-parse --short HEAD

In [ ]:
# @title 3) Install Dependencies (Colab CUDA)
import importlib.util

!pip -q install --upgrade pip setuptools wheel
!pip -q install -r requirements_cuda.txt

if importlib.util.find_spec('xformers') is None:
    !pip -q install xformers

print('Dependency install complete.')

In [ ]:
# @title 4) Runtime Optimization Defaults
import multiprocessing as mp

GPU_NAME = torch.cuda.get_device_name(0).lower()
IS_T4 = 't4' in GPU_NAME
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

# Conservative defaults for Colab free-tier stability.
CONFIG = 'colab_max' if IS_T4 else 'colab'  # @param ['nano', 'small', 'medium', 'colab', 'colab_max']
BATCH_SIZE = 1  # @param {type:'integer'}
GRAD_ACCUM = 16  # @param {type:'integer'}
NUM_WORKERS = min(4, max(2, (mp.cpu_count() or 2) // 2))  # @param {type:'integer'}
PREFETCH_FACTOR = 2  # @param {type:'integer'}
MAX_STEPS = 50000  # @param {type:'integer'}
SAVE_INTERVAL = 500  # @param {type:'integer'}
LOG_INTERVAL = 1  # @param {type:'integer'}
RESUME = True  # @param {type:'boolean'}
GRAD_CHECKPOINT = True  # @param {type:'boolean'}
COMPILE = True  # @param {type:'boolean'}

print(f'GPU={GPU_NAME}, VRAM={VRAM_GB:.1f}GB')
print(f'NUM_WORKERS={NUM_WORKERS}, PREFETCH_FACTOR={PREFETCH_FACTOR}, CONFIG={CONFIG}')

## 2) Pretraining

In [ ]:
# @title 5) Prepare Pretraining Data
# Supported by your codebase: .txt .md .pdf .epub .srt .vtt .ass .ssa
print('Preparing data from:', DATA_DIR)
!python dataset.py --data_dir "{DATA_DIR}"

In [ ]:
# @title 6) Run Pretraining
resume_flag = '--resume' if RESUME else ''
gc_flag = '--grad_checkpoint' if GRAD_CHECKPOINT else ''
compile_flag = '--compile' if COMPILE else ''

!python train.py \
  --config {CONFIG} \
  --data_dir "{DATA_DIR}" \
  --checkpoint_dir "{PRETRAIN_DIR}" \
  --batch_size {BATCH_SIZE} \
  --grad_accum {GRAD_ACCUM} \
  --max_steps {MAX_STEPS} \
  --save_interval {SAVE_INTERVAL} \
  --log_interval {LOG_INTERVAL} \
  --num_workers {NUM_WORKERS} \
  --prefetch_factor {PREFETCH_FACTOR} \
  {resume_flag} \
  {gc_flag} \
  {compile_flag}

## 3) Supervised Fine-Tuning (SFT)

In [ ]:
# @title 7) Run SFT
PRETRAINED_CHECKPOINT = f'{PRETRAIN_DIR}/step_500.pt'  # @param {type:'string'}
SFT_DATA = 'sft_data/train.jsonl'  # @param {type:'string'}
SFT_VAL_DATA = ''  # @param {type:'string'}
SFT_BATCH_SIZE = 1  # @param {type:'integer'}
SFT_GRAD_ACCUM = 16  # @param {type:'integer'}
MAX_SFT_STEPS = 2000  # @param {type:'integer'}
SFT_LR = 2e-5  # @param {type:'number'}
SFT_WARMUP = 50  # @param {type:'integer'}
SFT_SAVE_INTERVAL = 200  # @param {type:'integer'}
SFT_EVAL_INTERVAL = 100  # @param {type:'integer'}
SFT_GRAD_CHECKPOINT = True  # @param {type:'boolean'}

assert os.path.exists(PRETRAINED_CHECKPOINT), f'Missing checkpoint: {PRETRAINED_CHECKPOINT}'
gc_flag = '--grad_checkpoint' if SFT_GRAD_CHECKPOINT else ''
val_flag = f'--val_data "{SFT_VAL_DATA}"' if SFT_VAL_DATA.strip() else ''

!python sft_train.py \
  --checkpoint "{PRETRAINED_CHECKPOINT}" \
  --data "{SFT_DATA}" \
  --output_dir "{SFT_DIR}" \
  --batch_size {SFT_BATCH_SIZE} \
  --grad_accum {SFT_GRAD_ACCUM} \
  --max_steps {MAX_SFT_STEPS} \
  --lr {SFT_LR} \
  --warmup_steps {SFT_WARMUP} \
  --save_interval {SFT_SAVE_INTERVAL} \
  --eval_interval {SFT_EVAL_INTERVAL} \
  --num_workers {NUM_WORKERS} \
  --prefetch_factor {PREFETCH_FACTOR} \
  {gc_flag} \
  {val_flag}

## 4) Interactive Chat

In [ ]:
# @title 8) Chat with the SFT Model
SFT_CHECKPOINT = f'{SFT_DIR}/sft_final.pt'  # @param {type:'string'}
assert os.path.exists(SFT_CHECKPOINT), f'Missing SFT checkpoint: {SFT_CHECKPOINT}'
!python chat.py --checkpoint "{SFT_CHECKPOINT}"

In [ ]:
# @title 9) Optional: Quick VRAM Cleanup
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Memory cleanup complete.')